In [1]:
import os

import numpy as np
import pandas as pd
import scipy as sp

# import tensorflow as tf
# from tensorflow.keras import Model, Input, losses, layers, optimizers

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
 
from sklearn.model_selection import train_test_split
# from sklearn.metrics import accuracy_score
# from sklearn.metrics import f1_score

from matplotlib import pyplot as plt
# from matplotlib.colors import BoundaryNorm
# import seaborn as sns

# import itertools
from tqdm.auto import trange, tqdm

from pathlib import Path

# from src.datasets import wilts
# from src.evaluation import data_benchmark
# from src.models import vae_keras
# from src.datasets import wilts#, dataset_utils

# print(tf.config.list_physical_devices('GPU'))
# tf.debugging.set_log_device_placement(True)

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# import tensorflow as tf
# for gpu in tf.config.list_physical_devices('GPU'):
#     tf.config.experimental.set_memory_growth(gpu, True)

In [2]:
# Returns a Test dataset that contains an equal amounts of each class
# y should contain only two classes 0 and 1
def TrainSplitEqualBinary(X, y, samples_n): #samples_n per class

    # Generate a random permutation of indices
    ixs = np.random.permutation(len(X))

    # Shuffle X and y using the same random indices
    X = X[ixs]
    y = y[ixs]
    
    indicesClass1 = []
    indicesClass2 = []
    
    for i in range(0, len(y)):
        if y[i] == 0 and len(indicesClass1) < samples_n:
            indicesClass1.append(i)
        elif y[i] == 1 and len(indicesClass2) < samples_n:
            indicesClass2.append(i)
            
        if len(indicesClass1) == samples_n and len(indicesClass2) == samples_n:
            break
    
    X_class1 = X[indicesClass1]
    X_class2 = X[indicesClass2]
    
    X_bal = np.concatenate((X_class1,X_class2), axis=0)
    
    #remove x_test from X
    X_rest = np.delete(X, indicesClass1 + indicesClass2, axis=0)
    
    Y_class1 = y[indicesClass1]
    Y_class2 = y[indicesClass2]
    
    y_bal = np.concatenate((Y_class1,Y_class2), axis=0)
    
    #remove y_test from y
    y_rest = np.delete(y, indicesClass1 + indicesClass2, axis=0)
    
    if (X_bal.shape[0] != 2 * samples_n or y_bal.shape[0] != 2 * samples_n):
        raise Exception("Problem with split 1!")
        
    if (X_bal.shape[0] + X_rest.shape[0] != X.shape[0] or y_bal.shape[0] + y_rest.shape[0] != y.shape[0]):
        raise Exception("Problem with split 2!")
    
    return X_bal, y_bal

## Balance data

In [18]:
# x_cat = np.load('/home/fjunpop/tab-ddpm-dp/data/wilt/X_cat_train_backup.npy', allow_pickle=True)
x_num = np.load('/home/fjunpop/tab-ddpm-dp/data/wilt/X_num_train_backup.npy', allow_pickle=True)
y = np.load('/home/fjunpop/tab-ddpm-dp/data/wilt/y_train_backup.npy', allow_pickle=True)

print(type(y))
print(np.unique(y, return_counts=True))
print(x_num.shape, y.shape)

x_train = x_num

x_train_bal, y_train_bal= TrainSplitEqualBinary(x_train, y, 6273) 

# x_new_cat = x_train_bal[:,:8]
x_new_num = x_train_bal.astype(np.float32)
y_new = y_train_bal


print(x_new_num.shape, y_new.shape)

# np.save('/home/fjunpop/tab-ddpm-dp/data/wilt/X_cat_train.npy', x_new_cat[:,:8]) 
np.save('/home/fjunpop/tab-ddpm-dp/data/wilt/X_num_train.npy', x_new_num) 
np.save('/home/fjunpop/tab-ddpm-dp/data/wilt/y_train.npy', y_new) 


# x_cat = np.load('/home/fjunpop/tab-ddpm-dp/data/wilt/X_cat_train.npy', allow_pickle=True)
x_num = np.load('/home/fjunpop/tab-ddpm-dp/data/wilt/X_num_train.npy', allow_pickle=True)
y = np.load('/home/fjunpop/tab-ddpm-dp/data/wilt/y_train.npy', allow_pickle=True)

print(x_num.shape, y.shape)


# print(x_train.shape)

# print(x_cat[:,-1])

<class 'numpy.ndarray'>
(array([0, 1]), array([2929,  167]))
(3096, 5) (3096,)


Exception: Problem with split 1!

## Restore default datasets

In [3]:
# x_cat = np.load('/home/fjunpop/tab-ddpm-dp/data/wilt/X_cat_train_backup.npy', allow_pickle=True)
x_num = np.load('/home/fjunpop/tab-ddpm-dp/data/wilt/X_num_train_backup.npy', allow_pickle=True)
y = np.load('/home/fjunpop/tab-ddpm-dp/data/wilt/y_train_backup.npy', allow_pickle=True)

print(x_cat.shape, x_num.shape, y.shape)


# np.save('/home/fjunpop/tab-ddpm-dp/data/wilt/X_cat_train.npy', x_cat) 
np.save('/home/fjunpop/tab-ddpm-dp/data/wilt/X_num_train.npy', x_num) 
np.save('/home/fjunpop/tab-ddpm-dp/data/wilt/y_train.npy', y) 

# x_cat = np.load('/home/fjunpop/tab-ddpm-dp/data/wilt/X_cat_train.npy', allow_pickle=True)
x_num = np.load('/home/fjunpop/tab-ddpm-dp/data/wilt/X_num_train.npy', allow_pickle=True)
y = np.load('/home/fjunpop/tab-ddpm-dp/data/wilt/y_train.npy', allow_pickle=True)

print(x_cat.shape, x_num.shape, y.shape)



FileNotFoundError: [Errno 2] No such file or directory: '/home/fjunpop/tab-ddpm-dp/data/wilt/X_cat_train_backup.npy'

## Check current sizes

In [3]:
# x_cat = np.load('/home/fjunpop/tab-ddpm-dp/data/wilt/X_cat_train.npy', allow_pickle=True)
x_num = np.load('/home/fjunpop/tab-ddpm-dp/data/wilt/X_num_train.npy', allow_pickle=True)
y = np.load('/home/fjunpop/tab-ddpm-dp/data/wilt/y_train.npy', allow_pickle=True)

print(x_num.shape, y.shape)


(3096, 5) (3096,)
